# Thêm Thư Viện

In [47]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [48]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

## Đọc data từ SQL Server

In [49]:
df_data_mon = pd.read_csv("./data_mon.csv")
print(df_data_mon)

      ID_Mon                                TenMon      KyHieu  FileDeCuong  \
0        120                   Triết học Mác-Lênin  LLCT130105          NaN   
1        121           Kinh tế chính trị Mác-Lênin  LLCT120205          NaN   
2        122             Chủ nghĩa xã hội khoa học  LLCT120405          NaN   
3        123                  Tư tưởng Hồ Chí Minh  LLCT120314          NaN   
4        124        Lịch sử Đảng Cộng sản Việt Nam  LLCT220514          NaN   
...      ...                                   ...         ...          ...   
1593    1713  Hệ thống cấp nước và xử lý nước thải  WSWT341022          NaN   
1594    1714       Chuyên đề Doanh nghiệp (QLVHHT)  SCIC421422          NaN   
1595    1715                TT Tốt nghiệp (QLVHHT)  ENPR441522          NaN   
1596    1716                  Khóa luận tốt nghiệp  THSI471622          NaN   
1597    1717                    Hệ thống Logistics  LOSY421722          NaN   

      ID_Khoa  ID_BoMon  ID_LoaiMon  TinhTrang  
0 

## Xử lý data

## Thêm dòng không xác định

In [ ]:
new_row = pd.DataFrame({
    'ID_Mon': [0],
    'TenMon': ['(Không xác định)'],
})
df_data_mon = pd.concat([df_data_mon, new_row], ignore_index=True) # Thêm vào dataset
df_data_mon = df_data_mon.sort_values(by='ID_Mon', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
df_data_mon = df_data_mon.drop(columns=["FileDeCuong", "ID_Khoa", "ID_BoMon", "ID_LoaiMon","TinhTrang"])
print(df_data_mon)

      ID_Mon                                TenMon      KyHieu
0          0                      (Không xác định)         NaN
1        120                   Triết học Mác-Lênin  LLCT130105
2        121           Kinh tế chính trị Mác-Lênin  LLCT120205
3        122             Chủ nghĩa xã hội khoa học  LLCT120405
4        123                  Tư tưởng Hồ Chí Minh  LLCT120314
...      ...                                   ...         ...
1594    1713  Hệ thống cấp nước và xử lý nước thải  WSWT341022
1595    1714       Chuyên đề Doanh nghiệp (QLVHHT)  SCIC421422
1596    1715                TT Tốt nghiệp (QLVHHT)  ENPR441522
1597    1716                  Khóa luận tốt nghiệp  THSI471622
1598    1717                    Hệ thống Logistics  LOSY421722

[1599 rows x 3 columns]


### Xử lý data rỗng hoặc " "

In [51]:
df_data_mon = df_data_mon.replace(np.nan, None)
df_data_mon = df_data_mon.replace('', None)
print(df_data_mon)

      ID_Mon                                TenMon      KyHieu
0          0                      (Không xác định)        None
1        120                   Triết học Mác-Lênin  LLCT130105
2        121           Kinh tế chính trị Mác-Lênin  LLCT120205
3        122             Chủ nghĩa xã hội khoa học  LLCT120405
4        123                  Tư tưởng Hồ Chí Minh  LLCT120314
...      ...                                   ...         ...
1594    1713  Hệ thống cấp nước và xử lý nước thải  WSWT341022
1595    1714       Chuyên đề Doanh nghiệp (QLVHHT)  SCIC421422
1596    1715                TT Tốt nghiệp (QLVHHT)  ENPR441522
1597    1716                  Khóa luận tốt nghiệp  THSI471622
1598    1717                    Hệ thống Logistics  LOSY421722

[1599 rows x 3 columns]


## Load data

### [Nếu cần] Clear bảng

In [52]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM DIM_Mon"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [53]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO DIM_Mon (ID_mon, Ten_mon, Ky_hieu) 
                VALUES (?, ?, ?)
               """
for index, row in df_data_mon.iterrows():
    values = (row['ID_Mon'], row['TenMon'], row['KyHieu'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()